# FormFactor (NASDAQ: FORM) 財務分析 — Step 1：資料擷取 → 清理 → CSV

這本 notebook 的流程：

| 步驟 | 做什麼 | 學到的能力 |
|---|---|---|
| 1. 擷取 (Extract) | 用 SEC EDGAR 的免費 API 抓 FormFactor 所有 XBRL 財報數字 | `requests`、API、JSON 結構 |
| 2. 攤平 (Flatten) | 把巢狀 JSON 轉成一張「長表格」 | 巢狀資料 → DataFrame |
| 3. 清理 (Clean) | 只留 10-K 年報、去重複、處理重編 (restatement)、對齊會計年度 | 資料清理的核心邏輯 |
| 4. 對應 (Map) | 把 XBRL 標籤轉成我們看得懂的指標（營收、毛利、FCF…） | 資料字典、備援欄位 |
| 5. 驗證 (Validate) | 用會計恆等式檢查資料是否正確 | 資料品質檢查 |
| 6. 輸出 (Export) | 存成 CSV，下一步載入 SQL | ETL 流程 |

> **ETL** = Extract, Transform, Load。這就是資料分析師每天在做的事。

## 0. 環境設定

SEC 規定：每個請求都要帶 **User-Agent**，內容是「你的名字 + email」，否則會被擋 (403)。
👉 **請把下面的名字和 email 改成你自己的。**

In [3]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ⚠️ 改成你自己的資訊（SEC 要求）
HEADERS = {"User-Agent": "Chen-Chih Hung cosby432@gmail.com"}

# FormFactor 在 SEC 的公司代碼 (CIK)，API 需要補零到 10 位數
CIK = "1039399"
CIK10 = CIK.zfill(10)          # -> '0001039399'

# 專案資料夾結構
BASE = Path.cwd()
RAW_DIR = BASE / "data" / "raw"
CLEAN_DIR = BASE / "data" / "clean"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("資料會存在:", BASE / "data")

資料會存在: C:\Users\User\Documents\FormFactor_Analysis\data


## 1. 擷取 (Extract)：呼叫 SEC API

SEC 的 **companyfacts** API 會一次回傳這家公司「所有申報過的 XBRL 財務數字」。
XBRL 是財報的機器可讀格式，每個會計科目都有標準標籤，例如：
- `RevenueFromContractWithCustomerExcludingAssessedTax` = 營收
- `NetIncomeLoss` = 淨利

我們先把原始 JSON **原封不動存下來**（好習慣：raw data 永遠保留一份，之後清理出錯可以重來，不用再打 API）。

In [6]:
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK10}.json"
resp = requests.get(url, headers=HEADERS, timeout=30)
print("HTTP 狀態碼:", resp.status_code)   # 200 = 成功, 403 = User-Agent 沒設好
resp.raise_for_status()

facts = resp.json()
raw_path = RAW_DIR / "formfactor_companyfacts.json"
raw_path.write_text(json.dumps(facts), encoding="utf-8")
print("公司:", facts["entityName"])
print("原始檔大小: %.1f MB" % (raw_path.stat().st_size / 1e6))

HTTP 狀態碼: 200
公司: FormFactor, Inc.
原始檔大小: 4.0 MB


### 1.1 先看懂 JSON 的結構

拿到新資料的第一件事：**先探索結構，不要急著處理。**

```
facts
 └─ "facts"
     ├─ "dei"      ← 公司基本資訊（股數、公司名稱…）
     └─ "us-gaap"  ← 美國會計準則的財務科目 ★ 我們要的
          └─ 科目名稱 (例: "NetIncomeLoss")
               ├─ "label", "description"
               └─ "units"
                    └─ "USD": [ {start, end, val, fy, fp, form, filed, frame, accn}, ... ]
```

In [7]:
print("第一層:", list(facts.keys()))
print("分類:", list(facts["facts"].keys()))
print("us-gaap 科目數量:", len(facts["facts"]["us-gaap"]))

# 看一個科目長什麼樣子
ni = facts["facts"]["us-gaap"]["NetIncomeLoss"]

print("\nLabel:", ni["label"])
print("單位:", list(ni["units"].keys()))
print("筆數:", len(ni["units"]["USD"]))
ni["units"]["USD"][:3]

第一層: ['cik', 'entityName', 'facts']
分類: ['dei', 'us-gaap', 'ffd']
us-gaap 科目數量: 520

Label: Net Income (Loss) Attributable to Parent
單位: ['USD']
筆數: 255


[{'start': '2008-12-28',
  'end': '2009-12-26',
  'val': -155653000,
  'accn': '0001039399-12-000006',
  'fy': 2011,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2012-02-21',
  'frame': 'CY2009'},
 {'start': '2009-12-27',
  'end': '2010-12-25',
  'val': -188286000,
  'accn': '0001039399-12-000006',
  'fy': 2011,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2012-02-21'},
 {'start': '2009-12-27',
  'end': '2010-12-25',
  'val': -188286000,
  'accn': '0001445305-13-000585',
  'fy': 2012,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2013-03-13',
  'frame': 'CY2010'}]

**每個欄位的意思**（很重要，清理時會用到）：

| 欄位 | 意思 |
|---|---|
| `start` / `end` | 這個數字涵蓋的期間。**損益表、現金流量表**有 start（一段期間）；**資產負債表**只有 end（某個時間點） |
| `val` | 數值 |
| `form` | 出自哪份文件：`10-K` 年報、`10-Q` 季報 |
| `filed` | 申報日期 |
| `fy` / `fp` | ⚠️ 這是**那份文件**的會計年度，不是這個數字的年度！（2024 年的 10-K 裡也會有 2022、2023 的比較數字） |
| `frame` | SEC 幫忙標的日曆期間，例如 `CY2023` |
| `accn` | 文件編號 |

## 2. 攤平 (Flatten)：巢狀 JSON → 長表格

「長表格 (long format)」= 每一列是一個數字，欄位描述它是什麼。這是最方便清理和放進 SQL 的格式。

In [8]:
rows = []
for taxonomy, concepts in facts["facts"].items():
    for concept, info in concepts.items():
        for unit, entries in info["units"].items():
            for e in entries:
                rows.append({
                    "taxonomy": taxonomy,
                    "concept": concept,
                    "label": info.get("label"),
                    "unit": unit,
                    **e,          # 把 start, end, val, fy, fp, form, filed... 展開成欄位
                })

df_raw = pd.DataFrame(rows)
df_raw.to_csv(RAW_DIR / "formfactor_facts_long_raw.csv", index=False)
print(df_raw.shape)
df_raw.head()

(24086, 13)


,taxonomy,concept,label,unit,end,val,accn,fy,fp,form,filed,frame,start
0,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2011-07-27,"50,849,589.00",0001104659-11-043177,"2,011.00",Q2,10-Q,2011-08-03,CY2011Q2I,NaN
1,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2011-10-20,"50,460,810.00",0001104659-11-058508,"2,011.00",Q3,10-Q,2011-10-27,CY2011Q3I,NaN
2,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2012-02-13,"49,574,715.00",0001039399-12-000006,"2,011.00",FY,10-K,2012-02-21,CY2011Q4I,NaN
3,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2012-04-26,"49,610,899.00",0001039399-12-000010,"2,012.00",Q1,10-Q,2012-05-01,CY2012Q1I,NaN
4,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2012-07-27,"49,987,469.00",0001039399-12-000013,"2,012.00",Q2,10-Q,2012-08-01,CY2012Q2I,NaN


In [ ]:
# 快速檢查：有哪些文件類型？每種有多少筆？
df_raw["form"].value_counts()

## 3. 清理 (Clean)

原始資料有幾個問題，要一個一個處理：

1. **混了季報和年報** → 只留 10-K
2. **10-K 裡有些是季度數字**（有些年報會附第四季資料） → 用期間長度篩掉
3. **同一個數字出現很多次**：2025 的 10-K 會重報 2024、2023 的數字當比較 → 要去重
4. **重編 (restatement)**：舊數字在新年報中被修正 → 保留**最新申報**的版本
5. **會計年度**：FormFactor 的年度結束在 12 月最後一個週六（52/53 週制），例如 2024-12-28。不能直接用 `fy` 欄位，要從 `end` 日期推算

### 3.1 只留 us-gaap + 美元 + 年報

In [9]:
df = df_raw.copy()
df = df[(df["taxonomy"] == "us-gaap") & (df["form"].isin(["10-K", "10-K/A"]))]

# 字串 → 日期型態（沒有 start 的會變 NaT = 日期版的缺值）
for col in ["start", "end", "filed"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print("篩選後筆數:", len(df))
df["unit"].value_counts()

篩選後筆數: 9211


unit
USD           8353
shares         445
USD/shares     280
pure            94
segment         25
positions        6
unit             4
Employee         4
Name: count, dtype: int64

### 3.2 分辨「期間數字」vs「時點數字」，並推算會計年度
- **flow（期間）**：有 start → 損益表、現金流量表。一年期 ≈ 365 天（52/53 週制會是 364 或 371 天）
- **instant（時點）**：沒有 start → 資產負債表

In [10]:
df["period_type"] = df["start"].isna().map({True: "instant", False: "flow"})
df["duration_days"] = (df["end"] - df["start"]).dt.days

# 看看 flow 的期間長度分布：大部分應該 ~364 或 ~371 天，少數 ~91 天是季度
df.loc[df["period_type"] == "flow", "duration_days"].value_counts().head(10)

duration_days
363.00    3816
90.00     1122
370.00     959
97.00       55
1.00         4
272.00       2
181.00       1
270.00       1
Name: count, dtype: int64

In [11]:
# 只留完整一年的 flow（350~380 天），instant 全留
is_annual_flow = (df["period_type"] == "flow") & df["duration_days"].between(350, 380)
is_instant = df["period_type"] == "instant"
df = df[is_annual_flow | is_instant].copy()

# 會計年度：年度結束月份在下半年 → 就是那一年；如果在 1~5 月結束 → 算前一年
# （FormFactor 都在 12 月底結束，但寫成通用規則，換其他公司也能用）
df["fiscal_year"] = df["end"].dt.year.where(df["end"].dt.month >= 6, df["end"].dt.year - 1)
df[["concept", "start", "end", "duration_days", "fiscal_year", "val", "form", "filed"]].head()

,concept,start,end,duration_days,fiscal_year,val,form,filed
78,AccountsPayableCurrent,NaT,2010-12-25,NaN,2010,"14,948,000.00",10-K,2012-02-21
81,AccountsPayableCurrent,NaT,2011-12-31,NaN,2011,"9,731,000.00",10-K,2012-02-21
85,AccountsPayableCurrent,NaT,2011-12-31,NaN,2011,"9,731,000.00",10-K,2013-03-13
89,AccountsPayableCurrent,NaT,2012-12-29,NaN,2012,"21,015,000.00",10-K,2013-03-13
93,AccountsPayableCurrent,NaT,2012-12-29,NaN,2012,"21,015,000.00",10-K,2014-03-07


### 3.3 資產負債表：只留「年度結束日」的時點

10-K 裡的時點數字不一定都在年底（例如股東權益變動表會有期初數字）。
做法：用**營收的年度結束日**當標準，只保留落在這些日期的資產負債表數字。

In [12]:
fy_end_dates = set(
    df.loc[(df["period_type"] == "flow")
           & (df["concept"] == "RevenueFromContractWithCustomerExcludingAssessedTax"), "end"]
)
print("年度結束日:", sorted(d.date() for d in fy_end_dates))

df = df[(df["period_type"] == "flow") | (df["end"].isin(fy_end_dates))].copy()

年度結束日: [datetime.date(2016, 12, 31), datetime.date(2017, 12, 30), datetime.date(2018, 12, 29), datetime.date(2019, 12, 28), datetime.date(2020, 12, 26), datetime.date(2021, 12, 25), datetime.date(2022, 12, 31), datetime.date(2023, 12, 30), datetime.date(2024, 12, 28), datetime.date(2025, 12, 27)]


### 3.4 去重複 + 處理重編

同一個 (科目, 會計年度) 可能出現 3 次（出現在連續三年的年報裡）。
**規則：保留最新申報 (`filed` 最晚) 的那筆**，因為如果有重編，最新的才是正確的。

先看看真的有沒有重編（同一年同科目，數字不一樣）：

In [13]:
key = ["concept", "unit", "fiscal_year", "period_type"]
n_versions = df.groupby(key)["val"].nunique()
restated = n_versions[n_versions > 1]
print(f"有 {len(restated)} 組 (科目, 年度) 在不同年報中數字不同（重編或重分類）")
restated.head(10)

有 124 組 (科目, 年度) 在不同年報中數字不同（重編或重分類）


concept                                                                              unit  fiscal_year  period_type
AccretionAmortizationOfDiscountsAndPremiumsInvestments                               USD   2010         flow           2
                                                                                           2011         flow           2
AccrualForTaxesOtherThanIncomeTaxesCurrent                                           USD   2016         instant        2
AccruedLiabilitiesCurrent                                                            USD   2017         instant        2
AccumulatedOtherComprehensiveIncomeLossForeignCurrencyTranslationAdjustmentNetOfTax  USD   2018         instant        2
AllowanceForDoubtfulAccountsReceivablePeriodIncreaseDecrease                         USD   2011         flow           2
                                                                                           2012         flow           2
AmortizationOfIntangibleAssets       

In [14]:
df = (df.sort_values("filed")
        .drop_duplicates(subset=key, keep="last")   # 保留最新申報
        .reset_index(drop=True))
print("去重後筆數:", len(df))
print("年度範圍:", df["fiscal_year"].min(), "~", df["fiscal_year"].max())

去重後筆數: 3316
年度範圍: 2009 ~ 2025


## 4. 對應 (Map)：XBRL 標籤 → 分析指標

不同公司（甚至同公司不同年份）可能用不同標籤表示同一件事。
例如銷貨成本可能是 `CostOfRevenue` 或 `CostOfGoodsAndServicesSold`。

做法：每個指標列出**多個候選標籤**，照順序找，第一個有資料的就用。這叫「備援欄位 (fallback)」。

先寫一個小工具，用關鍵字搜尋有哪些標籤：

In [15]:
def find_concepts(keyword):
    # 用關鍵字搜尋科目名稱，回傳有幾年的資料
    m = df[df["concept"].str.contains(keyword, case=False)]
    return (m.groupby("concept")["fiscal_year"]
             .agg(["min", "max", "count"])
             .sort_values("count", ascending=False))

find_concepts("CostOf")

,min,max,count
concept,,,
CostOfGoodsAndServicesSold,2009,2025,17
CostOfGoodsAndServicesSoldAmortization,2018,2024,7


In [16]:
# 你可以試試其他關鍵字，例如：
find_concepts("Research")

,min,max,count
concept,,,
ResearchAndDevelopmentExpense,2009,2025,17
IncomeTaxReconciliationTaxCreditsResearch,2010,2024,15


In [17]:
METRICS = {
    # ---- 損益表 (Income Statement) ----
    "revenue":            ("IS", ["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues", "SalesRevenueNet"]),
    "cost_of_revenue":    ("IS", ["CostOfRevenue", "CostOfGoodsAndServicesSold", "CostOfGoodsSold"]),
    "gross_profit":       ("IS", ["GrossProfit"]),
    "rd_expense":         ("IS", ["ResearchAndDevelopmentExpense"]),
    "sga_expense":        ("IS", ["SellingGeneralAndAdministrativeExpense"]),
    "operating_income":   ("IS", ["OperatingIncomeLoss"]),
    "income_tax":         ("IS", ["IncomeTaxExpenseBenefit"]),
    "net_income":         ("IS", ["NetIncomeLoss"]),
    "eps_diluted":        ("IS", ["EarningsPerShareDiluted"]),
    "shares_diluted":     ("IS", ["WeightedAverageNumberOfDilutedSharesOutstanding"]),
    # ---- 資產負債表 (Balance Sheet) ----
    "cash":               ("BS", ["CashAndCashEquivalentsAtCarryingValue"]),
    "st_investments":     ("BS", ["MarketableSecuritiesCurrent", "AvailableForSaleSecuritiesDebtSecuritiesCurrent", "ShortTermInvestments"]),
    "accounts_receivable":("BS", ["AccountsReceivableNetCurrent"]),
    "inventory":          ("BS", ["InventoryNet"]),
    "current_assets":     ("BS", ["AssetsCurrent"]),
    "total_assets":       ("BS", ["Assets"]),
    "current_liabilities":("BS", ["LiabilitiesCurrent"]),
    "total_liabilities":  ("BS", ["Liabilities"]),
    "long_term_debt":     ("BS", ["LongTermDebtNoncurrent", "LongTermDebt"]),
    "total_equity":       ("BS", ["StockholdersEquity"]),
    "liab_and_equity":    ("BS", ["LiabilitiesAndStockholdersEquity"]),
    "goodwill":           ("BS", ["Goodwill"]),
    # ---- 現金流量表 (Cash Flow) ----
    "operating_cash_flow":("CF", ["NetCashProvidedByUsedInOperatingActivities"]),
    "capex":              ("CF", ["PaymentsToAcquirePropertyPlantAndEquipment"]),
    "depreciation":       ("CF", ["Depreciation", "DepreciationDepletionAndAmortization", "DepreciationAmortizationAndAccretionNet"]),
    "stock_comp":         ("CF", ["ShareBasedCompensation", "AllocatedShareBasedCompensationExpense"]),
    "buybacks":           ("CF", ["PaymentsForRepurchaseOfCommonStock"]),
    "acquisitions":       ("CF", ["PaymentsToAcquireBusinessesNetOfCashAcquired"]),
}

In [18]:
records, report = [], []
for metric, (statement, candidates) in METRICS.items():
    # 逐年挑選：每一年用「第一個有資料的候選標籤」
    # （同一公司不同年份可能換標籤，所以逐年處理比整個指標挑一個標籤更完整）
    sub = df[df["concept"].isin(candidates)].copy()
    if sub.empty:
        report.append((metric, "❌ 找不到", ""))
        continue
    sub["priority"] = sub["concept"].map({c: i for i, c in enumerate(candidates)})
    sub = sub.sort_values("priority").drop_duplicates("fiscal_year", keep="first")
    sub["metric"], sub["statement"] = metric, statement
    records.append(sub)
    report.append((metric, "✅", ", ".join(sub["concept"].unique())))

fin_long = pd.concat(records, ignore_index=True)
pd.DataFrame(report, columns=["metric", "狀態", "使用的標籤"])

,metric,狀態,使用的標籤
0,revenue,✅,RevenueFromContractWithCustomerExcludingAssess...
1,cost_of_revenue,✅,CostOfGoodsAndServicesSold
2,gross_profit,✅,GrossProfit
3,rd_expense,✅,ResearchAndDevelopmentExpense
4,sga_expense,✅,SellingGeneralAndAdministrativeExpense
5,operating_income,✅,OperatingIncomeLoss
6,income_tax,✅,IncomeTaxExpenseBenefit
7,net_income,✅,NetIncomeLoss
8,eps_diluted,✅,EarningsPerShareDiluted
9,shares_diluted,✅,WeightedAverageNumberOfDilutedSharesOutstanding


### 4.1 補上缺的指標（用會計公式推算）

有些公司不申報某些總數（例如很多公司沒有 `Liabilities` 總負債）。可以用會計恆等式推算：
- 總負債 = 負債與權益合計 − 股東權益
- 毛利 = 營收 − 銷貨成本

推算出來的數字標記 `concept = "derived"`，讓之後的人知道這不是原始申報值。

In [19]:
wide = fin_long.pivot_table(index="fiscal_year", columns="metric", values="val")

def fill_derived(metric, formula_values, statement):
    # 只補「原本缺值」的年份
    global fin_long
    if metric not in wide:
        wide[metric] = float("nan")
    missing = wide[metric].isna() & formula_values.notna()
    if missing.any():
        wide.loc[missing, metric] = formula_values[missing]
        add = pd.DataFrame({"fiscal_year": wide.index[missing], "metric": metric,
                            "val": formula_values[missing].values, "concept": "derived",
                            "statement": statement})
        fin_long = pd.concat([fin_long, add], ignore_index=True)
        print(f"{metric}: 推算補上 {missing.sum()} 年")

fill_derived("total_liabilities", wide["liab_and_equity"] - wide["total_equity"], "BS")
fill_derived("gross_profit", wide["revenue"] - wide["cost_of_revenue"], "IS")

## 5. 驗證 (Validate)

**永遠不要相信沒驗證過的資料。** 三個檢查：
1. 資產 = 負債 + 權益（會計恆等式，一定要成立）
2. 毛利 = 營收 − 成本
3. 跟公開資訊對照：FormFactor 2024 年營收約 **$763.6M**、2025 年約 **$785.0M**

In [20]:
check = pd.DataFrame({
    "資產-負債-權益": wide["total_assets"] - wide["total_liabilities"] - wide["total_equity"],
    "毛利-(營收-成本)": wide["gross_profit"] - (wide["revenue"] - wide["cost_of_revenue"]),
})
print("差異應該都是 0（或非常接近 0）：")
check

差異應該都是 0（或非常接近 0）：


,資產-負債-權益,毛利-(營收-成本)
fiscal_year,,
2009,NaN,0.00
2010,NaN,0.00
2011,NaN,0.00
2012,NaN,0.00
2013,NaN,0.00
2014,NaN,0.00
2015,NaN,0.00
2016,0.00,0.00
2017,0.00,0.00


In [21]:
# 以百萬美元顯示主要數字，和公開資訊對照
(wide[["revenue", "gross_profit", "operating_income", "net_income", "operating_cash_flow"]] / 1e6).round(1)

metric,revenue,gross_profit,operating_income,net_income,operating_cash_flow
fiscal_year,,,,,
2009,135.30,0.80,-145.20,-155.70,NaN
2010,188.60,-2.30,-197.20,-188.30,NaN
2011,169.30,21.00,-70.40,-66.00,NaN
2012,178.50,25.30,-64.10,-35.50,NaN
2013,231.50,42.30,-58.80,-57.70,NaN
2014,268.50,77.40,-20.60,-19.20,NaN
2015,282.40,85.70,-4.10,-1.50,NaN
2016,383.90,102.70,-47.90,-6.60,NaN
2017,548.40,215.60,46.30,40.90,86.30


## 6. 輸出 (Export) CSV

輸出三個檔案，下一步會載入 SQL 資料庫：

| 檔案 | 格式 | 用途 |
|---|---|---|
| `formfactor_financials_long.csv` | 長表格（年度 × 指標 一列） | 載入 SQL 的主表 ★ |
| `formfactor_financials_wide.csv` | 寬表格（一年一列） | Excel 瀏覽、畫圖 |
| `metric_dictionary.csv` | 指標說明 | 資料字典 |

In [22]:
# 只留最近 10 個會計年度

last_fy = int(wide.index.max())
keep_years = range(last_fy - 9, last_fy + 1)

out_long = (fin_long[fin_long["fiscal_year"].isin(keep_years)]
            [["fiscal_year", "statement", "metric", "val", "concept", "end", "filed", "accn"]]
            .rename(columns={"val": "value", "end": "period_end", "filed": "filed_date"})
            .sort_values(["statement", "metric", "fiscal_year"]))
out_long["ticker"] = "FORM"
out_long = out_long[["ticker"] + [c for c in out_long.columns if c != "ticker"]]
out_long.to_csv(CLEAN_DIR / "formfactor_financials_long.csv", index=False)

out_wide = wide.loc[wide.index.isin(keep_years)].reindex(columns=list(METRICS.keys()))
out_wide.to_csv(CLEAN_DIR / "formfactor_financials_wide.csv")

pd.DataFrame(
    [(m, s, " | ".join(c)) for m, (s, c) in METRICS.items()],
    columns=["metric", "statement", "xbrl_candidates"],
).to_csv(CLEAN_DIR / "metric_dictionary.csv", index=False)

print("✅ 完成！輸出：")
for p in sorted(CLEAN_DIR.glob("*.csv")):
    print("  ", p.name, f"({len(pd.read_csv(p))} 列)")

✅ 完成！輸出：
   formfactor_financials_long.csv (278 列)
   formfactor_financials_wide.csv (10 列)
   metric_dictionary.csv (28 列)


## ✏️ 練習題（自己試試看）

1. 用 `find_concepts("Segment")` 或 `find_concepts("Revenue")` 看看 FormFactor 有沒有申報**分部營收**（探針卡 Probe Cards vs 系統 Systems）
2. 在 `METRICS` 裡新增一個指標：`"interest_expense": ("IS", ["InterestExpense"])`，重跑看看有沒有資料
3. 把 `CIK` 改成競爭對手的，例如 **Teradyne (97210)**、**Cohu (21535)**，整本重跑 — 這就是 Step 3 同業比較的基礎

**下一步 → `02_sql_analysis.ipynb`**：把 CSV 載入 SQLite，用 SQL 算毛利率、ROE、自由現金流、成長率。